In [22]:
import pandas as pd
# from sklearn.model_selection import train_test_split
df = pd.read_csv("dataset_splits2.csv")
vidnames = list(set(df["video_path"]))
real_vids = [name for name in vidnames if "original" in name]
fake_vids = [name for name in vidnames if "original" not in name]

train_vids = real_vids[0:5] + fake_vids[0:5]
test_vids = [real_vids[6], fake_vids[6]]

train_frames = df.loc[[vid in train_vids for vid in df["video_path"]]]
test_frames = df.loc[[vid in test_vids for vid in df["video_path"]]]

print(train_frames.iloc[500, -1])

1


In [27]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

class FrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """
        Returns a clip tensor of shape (3, T, H, W) = (3, 10, 500, 500).
        The DataLoader will stack these into (B, 3, T, H, W).
        """
        vidpath = self.df.iloc[idx, 0]
        tensors = []
        for i in range(10):
            fname = self.df.iloc[idx, i + 1]  # cols 1–10 are frame_0–frame_9
            path = os.path.join(vidpath, fname)
            img = Image.open(path).convert('RGB').resize((500, 500))
            tensors.append(self.transform(img)) # (3, 500, 500)
        clip = torch.stack(tensors, dim=1).to(dtype=torch.float32) # (3, T, 500, 500)
        label = torch.Tensor([self.df.iloc[idx, -1]])
        return clip, label


train_dataset = FrameDataset(train_frames)
train_loader = DataLoader(train_dataset, batch_size=4)
test_dataset = FrameDataset(test_frames)


cuda


In [33]:
def test_model(model, data, device='cpu', do_print=False):
    model.to(device=device)
    model.eval()
    total = 0
    correct = 0
    labels = ["Real", "Fake"]
    record = []
    for i, (frames, label) in enumerate(data):
        frames = frames.unsqueeze(0).to(device) # add batch dimension
        output = model(frames)[0] # since batch size is 1, we will get only one output
        y_pred = torch.round(output).item()
        y_actual = label
        record.append((y_pred, y_actual))
        if y_actual == y_pred:
            correct = correct+1
        total = total+1
    if do_print:  
        print(f"Accuracy = {correct / total * 100:.4f} %")
        for i in range(2):
            tps = sum([1 for x in record if x[0] == i and x[1] == i])
            fps = sum([1 for x in record if x[0] == i and x[1] != i])
            fns = sum([1 for x in record if x[0] != i and x[1] == i])
            tns = sum([1 for x in record if x[0] != i and x[1] != i])
            guesses = [x[0] for x in record if x[1] == i]
            # represents what the model guessed when this was the actual label
            counts = [guesses.count(j) for j in range(2)]
            if tps == 0:
                precision = 0.0
                recall = 0.0
                f_score = 0.0
            else:
                precision = tps / (tps + fps)
                recall = tps / (tps + fns)
                f_score = 2 * precision * recall / (precision + recall)
            print(f"  {labels[i]} ({i}):\n    F Score: {f_score:.4f}\n    Guesses: {counts}\n    Precision: {precision:.4f}\n    Recall: {recall:.4f}")
            if i == 1:
                print(f"\tPred Pos.\tPred Neg.\n\tReal Pos.\t{tps}\t{fns}\n\tReal Neg.\t{fps}\t{tns}")
    return correct / total * 100    

In [37]:
import pickle
from tqdm import tqdm
def train_model(model, train_loader, criterion, optimizer, num_epochs, test_data, device='cpu', output_dir=None):
    model.to(device)
    losses = []
    vals = []
    best_val = -1
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1} / {num_epochs}")
        for frames, labels in progress_bar:
            # Moving the data to GPU if available
            frames, labels = frames.to(device=device), labels.to(device=device)
            optimizer.zero_grad()
            outputs = model(frames)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            losses.append(loss.item())
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")
        val = test_model(model, test_data, device)
        vals.append(val)
        if output_dir:
            with open(os.path.join(output_dir, "stats.pkl"), "wb") as f:
                pickle.dump({"epoch": epoch + 1, "loss": losses, "val": vals}, f)
        if val > best_val:
            print(f" ! New best val: {val:.4f} !")
            best_val = val
            test_model(model, test_data, device, do_print=True)
#         elif epoch % 20 == 0:
#             test_model(model, test_data, device, do_print=True)
    return losses

In [38]:
from classifier_model import AIClassifier
lr = .0005
epochs = 3

criterion = nn.BCELoss().to(device=device)
model = AIClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)


train_model(model, train_loader, criterion, optimizer, epochs, test_dataset, device=device, output_dir="output")

Epoch 1 / 3: 100%|██████████| 164/164 [06:58<00:00,  2.55s/it]


Epoch 1/3, Loss: 6850.6389
 ! New best val: 49.0909 !
Accuracy = 49.0909 %
  Real (0):
    F Score: 0.6585
    Guesses: [54, 0]
    Precision: 0.4909
    Recall: 1.0000
  Fake (1):
    F Score: 0.0000
    Guesses: [56, 0]
    Precision: 0.0000
    Recall: 0.0000
	Pred Pos.	Pred Neg.
	Real Pos.	0	56
	Real Neg.	0	54


Epoch 2 / 3: 100%|██████████| 164/164 [06:53<00:00,  2.52s/it]


Epoch 2/3, Loss: 6850.0000


Epoch 3 / 3: 100%|██████████| 164/164 [06:57<00:00,  2.55s/it]


Epoch 3/3, Loss: 6850.0000


[0.6388576030731201,
 1.109257027565036e-06,
 2.4951466267479974e-14,
 4.024901725350344e-21,
 8.74619375823105e-29,
 5.984540922372116e-34,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 50.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0,
 100.0